In [1]:
import asyncio
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled

# Step 1: Disable tracing (not needed for local models)
set_tracing_disabled(True)

# Step 2: Create an OpenAI-compatible client pointing to Ollama
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",   # Ollama's OpenAI-compatible endpoint
    api_key="ollama"                        # Dummy key — Ollama doesn't need one
)

# Step 3: Wrap it in the SDK's model class
local_model = OpenAIChatCompletionsModel(
    model="qwen2.5:7b",         # Must match what you pulled with `ollama pull`
    openai_client=ollama_client
)

# Step 4: Create agent with local model
agent = Agent(
    name="Local Assistant",
    instructions="You are a helpful assistant running locally via Ollama. Be concise.",
    model=local_model             # <-- This is the key line!
)

# Step 5: Run it
result = await Runner.run(agent, "What is the Agents SDK in one paragraph?")
print(result.final_output)

The Agents SDK is a software development kit that enables developers to integrate various AI and automated agent functionalities into applications, allowing for the creation of more intelligent and interactive user experiences through natural language processing, task automation, and dialogue management capabilities.


In [2]:
from openai import AsyncOpenAI
from agents import (
    Agent, Model, ModelProvider,
    OpenAIChatCompletionsModel, Runner,
    set_tracing_disabled
)

set_tracing_disabled(True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "qwen2.5:7b"


class OllamaProvider(ModelProvider):
    """Custom ModelProvider that routes all model requests to Ollama."""
    
    def __init__(self, model_name: str = OLLAMA_MODEL):
        self.model_name = model_name
        self.client = AsyncOpenAI(
            base_url=OLLAMA_BASE_URL,
            api_key="ollama"
        )
    
    def get_model(self, model_name: str | None = None) -> Model:
        return OpenAIChatCompletionsModel(
            model=model_name or self.model_name,
            openai_client=self.client
        )


# Now create agents WITHOUT specifying model each time
ollama = OllamaProvider()

agent = Agent(
    name="Local Agent",
    instructions="You are a local AI assistant. Be helpful and concise.",
    model=ollama.get_model()  # Clean and reusable!
)

result = await Runner.run(agent, "Explain what Ollama is in 2 sentences.")
print(result.final_output)

Ollama is an AI platform designed to enhance natural language processing tasks. It uses advanced models to understand and generate human-like text, making interactions more intuitive and effective.


In [3]:
# !pip install litellm

from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(True)

agent = Agent(
    name="LiteLLM Agent",
    instructions="You are a helpful assistant.",
    # IMPORTANT: prepend 'ollama_chat/' — tells LiteLLM to use Ollama's chat API
    model=LitellmModel(model="ollama_chat/qwen2.5:7b", base_url="http://localhost:11434")
)

result = await Runner.run(agent, "Hello! What model are you?")
print(result.final_output)

I'm an AI assistant created by Alibaba Cloud, and I go by the name Qwen. How can I assist you today?


In [5]:
# === ollama_setup.py — Copy this to your projects ===

from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

# Disable tracing for local models
set_tracing_disabled(True)


def get_ollama_model(
    model_name: str = "qwen2.5:7b",
    base_url: str = "http://localhost:11434/v1"
):
    """Create an Ollama-backed model for the Agents SDK.
    
    Args:
        model_name: Ollama model name (must be pulled first)
        base_url: Ollama server URL
    
    Returns:
        OpenAIChatCompletionsModel ready to use with Agent(model=...)
    """
    client = AsyncOpenAI(base_url=base_url, api_key="ollama")
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)


# Usage:
# from ollama_setup import get_ollama_model
# agent = Agent(name="My Agent", instructions="...", model=get_ollama_model())
# agent = Agent(name="My Agent", instructions="...", model=get_ollama_model("llama3.1:8b"))

print("ollama_setup module ready")

ollama_setup module ready


In [4]:
from pydantic import BaseModel, Field
from typing import Literal
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled


set_tracing_disabled(True)

# Local model — sensitive HR data NEVER leaves the machine
local_model = OpenAIChatCompletionsModel(
    model="qwen2.5:7b",
    openai_client=AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
)


class ResumeAnalysis(BaseModel):
    name: str = Field(description="Candidate name")
    years_experience: int = Field(description="Total years of experience")
    top_skills: list[str] = Field(description="Top 3-5 relevant skills")
    fit_score: int = Field(description="Job fit score 1-100")
    fit_level: Literal["strong", "moderate", "weak"] = Field(description="Overall fit")
    summary: str = Field(description="2-sentence summary for hiring manager")


resume_agent = Agent(
    name="Resume Analyzer",
    instructions="""
    You are an HR AI assistant that analyzes resumes against job requirements.
    
    JOB: Senior Python Developer
    REQUIREMENTS: 5+ years Python, FastAPI/Django, SQL, cloud experience (AWS/GCP)
    NICE TO HAVE: AI/ML, Docker, Kubernetes, system design
    
    Analyze the resume text and score the candidate's fit honestly.
    """,
    model=local_model,
    output_type=ResumeAnalysis
)

# Simulate a resume
resume_text = """
Ahmed Hassan — Lahore, Pakistan
7 years experience in software development.
Skills: Python, FastAPI, PostgreSQL, Redis, Docker, AWS (EC2, Lambda, S3)
Previous: Lead Developer at SAI Lab (3 years), Senior Dev at SAI (4 years)
Projects: Built a real-time analytics pipeline processing 1M events/day.
Education: BS Computer Science, UAF
"""

result = await Runner.run(resume_agent, resume_text)
r = result.final_output

print(f"{r.name}")
print(f"Experience: {r.years_experience} years")
print(f"Skills: {', '.join(r.top_skills)}")
print(f"Fit Score: {r.fit_score}/100 ({r.fit_level})")
print(f"Summary: {r.summary}")

Ahmed Hassan
Experience: 7 years
Skills: Python, FastAPI, PostgreSQL, Redis, Docker, AWS
Fit Score: 85/100 (strong)
Summary: Ahmed has a strong background in Python with relevant experience in FastAPI and has extensive cloud experience with AWS. He also possesses Docker knowledge which closely aligns with the job requirements.
